In [2]:
from dscribe.descriptors import SOAP
from ase import Atoms
from ase.data import atomic_numbers
import numpy as np
from tqdm import tqdm
# Function to read .xyz files
def read_xyz(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
        num_atoms = int(lines[0])
        atom_data = lines[2:2 + num_atoms]

        atoms = []
        coords = []

        for line in atom_data:
            parts = line.split()
            atoms.append(parts[0])  # atom symbol
            coords.append([float(x) for x in parts[1:4]])  # x, y, z

        return atoms, coords

# Load molecules from files
molecule = []
for i in range(1, 20001):
    file_path = fr"structures_train\mol_{i}.xyz"
    atoms, coords = read_xyz(file_path)
    molecule.append(Atoms(symbols=atoms, positions=coords))

# Get unique atomic numbers
species = list(set(atomic_numbers[atom.symbol] for mol in molecule for atom in mol))

# Create SOAP descriptor with average='inner'
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=8.0,
    n_max=11,
    l_max=11,
    average='inner',    # returns fixed-length vector for each molecule
    sparse=False
)

# Generate SOAP descriptors for all molecules
X = np.array([soap.create(mol) for mol in tqdm(molecule ,  desc="Reading molecules")])

# Check shapes
print("SOAP descriptor shape for first molecule:", X[0].shape)
print("Total molecules processed:", len(X))


Reading molecules: 100%|██████████| 20000/20000 [00:41<00:00, 480.47it/s]


SOAP descriptor shape for first molecule: (18480,)
Total molecules processed: 20000


In [3]:
import pandas as pd

# Load target values
df = pd.read_csv("train.csv")
y = df["dipole_moment"].values


In [4]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


# Load target values
df = pd.read_csv("train.csv")
y = df["dipole_moment"].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the feature matrix X and target y
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Set XGBoost parameters
params = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "verbosity": 0,
    "tree_method": "gpu_hist",
    "gpu_id": 0
}

# Create evaluation watchlist
watchlist = [(dtrain, "train"), (dtest, "eval")]

# Custom training progress callback
class TQDMCallback(xgb.callback.TrainingCallback):
    def __init__(self, total_rounds):
        self.pbar = tqdm(total=total_rounds, desc="Training Progress", unit="iter")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        self.pbar.close()
        return model  # ⚠️ This is required to avoid the after_training error

# Number of boosting rounds
num_round = 200

# Train model with TQDM progress
model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=num_round,
    evals=watchlist,
    callbacks=[TQDMCallback(num_round)]
)

# Predict and evaluate
y_pred = model.predict(dtest)
mae = mean_absolute_error(y_test, y_pred)
print(f"\nMean Absolute Error: {mae:.5f}")

# Define a custom "accuracy" (for regression)
accuracy = 1 - (mae / np.mean(y_test))
print(f"Custom Accuracy: {accuracy:.4f}")


Training Progress:   0%|          | 0/200 [00:00<?, ?iter/s]e:\Hackrush\venv310\lib\site-packages\xgboost\training.py:183: UserWarning: [04:03:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  bst.update(dtrain, iteration=i, fobj=obj)


[0]	train-mae:0.97535	eval-mae:0.98864


Training Progress:   0%|          | 1/200 [00:03<10:00,  3.02s/iter]

[1]	train-mae:0.79941	eval-mae:0.82010


Training Progress:   1%|          | 2/200 [00:12<22:25,  6.79s/iter]

...

[198]	train-mae:0.05264	eval-mae:0.41492


Training Progress: 100%|█████████▉| 199/200 [03:02<00:00,  1.51iter/s]

[199]	train-mae:0.05221	eval-mae:0.41495


Training Progress: 100%|██████████| 200/200 [03:02<00:00,  1.09iter/s]



Mean Absolute Error: 0.41495
Custom Accuracy: 0.8394


In [5]:
from bayes_opt import BayesianOptimization
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import numpy as np
import pandas as pd



# Extract feature importance
importance = model.get_score(importance_type='weight')
importance = sorted(importance.items(), key=lambda x: x[1], reverse=True)

# Get all feature indices sorted by importance
all_features = [int(k.replace('f', '')) for k, v in importance]

# Define function to optimize
def optimize_top_n_features(top_n_features):
    top_n_features = int(top_n_features)
    
    selected_features = all_features[:top_n_features]
    X_train_selected = X_train[:, selected_features]
    X_test_selected = X_test[:, selected_features]
    
    dtrain_selected = xgb.DMatrix(X_train_selected, label=y_train)
    dtest_selected = xgb.DMatrix(X_test_selected, label=y_test)
    
    model_selected = xgb.train(
        params=params,
        dtrain=dtrain_selected,
        num_boost_round=200,
        evals=[(dtest_selected, "eval")],
        verbose_eval=False
    )
    
    preds = model_selected.predict(dtest_selected)
    mae = mean_absolute_error(y_test, preds)
    return -mae  # Negative because we want to maximize accuracy (minimize MAE)

# Bayesian Optimization
optimizer = BayesianOptimization(
    f=optimize_top_n_features,
    pbounds={"top_n_features": (1000, min(15000, len(all_features)))},
    random_state=42,
    verbose=2
)

optimizer.maximize(
    init_points=5,
    n_iter=15
)

print("\nBest result:", optimizer.max)


|   iter    |  target   | top_n_... |
-------------------------------------
| 1         | -0.4126   | 2.366e+03 |
| 2         | -0.417    | 4.468e+03 |
| 3         | -0.4199   | 3.67e+03  |
| 4         | -0.4096   | 3.184e+03 |
| 5         | -0.4122   | 1.569e+03 |
| 6         | -0.4096   | 3.185e+03 |
| 7         | -0.4165   | 2.785e+03 |
| 8         | -0.411    | 1.957e+03 |
| 9         | -0.4129   | 1.127e+03 |
| 10        | -0.4103   | 2.147e+03 |
| 11        | -0.4109   | 1.353e+03 |
| 12        | -0.4138   | 1.782e+03 |
| 13        | -0.4123   | 4.11e+03  |
| 14        | -0.4117   | 3.345e+03 |
| 15        | -0.4074   | 3.043e+03 |
| 16        | -0.414    | 2.971e+03 |
| 17        | -0.4107   | 3.092e+03 |
| 18        | -0.4063   | 2.073e+03 |
| 19        | -0.4136   | 2.032e+03 |
| 20        | -0.4085   | 2.097e+03 |

Best result: {'target': np.float64(-0.40630402200668714), 'params': {'top_n_features': np.float64(2072.6720461608793)}}


In [1]:
xgb.plot_importance(model, max_num_features=20)


NameError: name 'xgb' is not defined